# 01 — Exploração Inicial do Dataset GitSkills

**Objetivo:** Carregar o dataset GitSkills e realizar uma exploração inicial para entender sua estrutura, volume e conteúdo — com foco em identificar skills relacionadas à **segurança no desenvolvimento com IA**.

**Dataset:** [GitSkills (Zenodo)](https://zenodo.org/records/21875637) · [Mirror HuggingFace](https://huggingface.co/datasets/mvaccargiu/gitskills)  
**Paper:** Destefanis et al. (2027). *GitSkills: A Dataset of Agent Skills on GitHub*. MSR '27. [arXiv:2608.10906](https://arxiv.org/abs/2608.10906)

---

### Pré-requisitos

Antes de executar este notebook, certifique-se de que:
1. O ambiente foi configurado: `make setup`
2. O dataset foi baixado: `make download`
3. O kernel **"Security Skills Analysis"** está selecionado

### Tabelas do Dataset

| Tabela | Registros | Conteúdo |
|--------|-----------|----------|
| `artifacts` | 3.797.117 | Um registro por SKILL.md: repositório, path, hash, texto, front matter |
| `artifact_siblings` | 7.264.865 | Scripts e arquivos de referência junto às skills |
| `repos` | 282.200 | Metadados dos repositórios |
| `mining_runs` | 7 | Log de proveniência |

## 1. Setup e Imports

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações de exibição
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 50)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Caminhos dos dados (Parquets baixados via `make download`)
DATA_DIR = Path('../data')

PARQUET_FILES = {
    'artifacts':         DATA_DIR / 'artifacts.parquet',
    'artifact_siblings': DATA_DIR / 'artifact_siblings.parquet',
    'repos':             DATA_DIR / 'repos.parquet',
    'mining_runs':       DATA_DIR / 'mining_runs.parquet',
}

# Verificar se os dados foram baixados
print('Status dos arquivos de dados:')
all_ok = True
for name, path in PARQUET_FILES.items():
    exists = path.exists()
    size = f'{path.stat().st_size / (1024**2):.1f} MB' if exists else '—'
    status = '✅' if exists else '❌'
    print(f'  {status} {name:25s} {size}')
    if not exists:
        all_ok = False

if not all_ok:
    print('\n⚠️  Dados incompletos! Execute: make download')

## 2. Carregamento do Dataset

Carregamos cada tabela a partir dos Parquets locais (baixados via `make download`).

> **Nota sobre memória:** A tabela `artifacts` tem ~3.8M linhas e pode ocupar bastante RAM. Se necessário, use `pd.read_parquet(..., columns=[...])` para selecionar apenas as colunas desejadas.

### 2.1 Tabelas menores — `repos`, `mining_runs`

In [ ]:
# Tabelas pequenas — carregam rápido e cabem na memória
df_repos = pd.read_parquet(PARQUET_FILES['repos'])
df_mining = pd.read_parquet(PARQUET_FILES['mining_runs'])

print(f'repos:       {df_repos.shape[0]:>10,} linhas × {df_repos.shape[1]} colunas')
print(f'mining_runs: {df_mining.shape[0]:>10,} linhas × {df_mining.shape[1]} colunas')

### 2.2 Tabela `artifacts` — carregamento parcial

A tabela `artifacts` é grande (~3.8M linhas). Para a exploração inicial, carregamos apenas os metadados (sem o campo `full_text` que é o mais pesado).

In [ ]:
# Primeiro: ver quais colunas existem (sem carregar dados)
import pyarrow.parquet as pq

pf = pq.ParquetFile(PARQUET_FILES['artifacts'])
all_columns = pf.schema.names
print(f'Colunas na tabela artifacts ({len(all_columns)}):')
for i, col in enumerate(all_columns):
    print(f'  {i+1:2d}. {col}')

In [ ]:
# Carregar metadados (sem full_text, para economizar memória)
# Ajuste esta lista conforme as colunas reais do dataset
META_COLUMNS = [c for c in all_columns if c not in ('full_text', 'content')]

print(f'Carregando {len(META_COLUMNS)} colunas de metadados...')
df_artifacts_meta = pd.read_parquet(PARQUET_FILES['artifacts'], columns=META_COLUMNS)
print(f'Carregado: {df_artifacts_meta.shape[0]:,} linhas × {df_artifacts_meta.shape[1]} colunas')
print(f'Memória: {df_artifacts_meta.memory_usage(deep=True).sum() / (1024**2):.1f} MB')

In [ ]:
# Carregar uma amostra COM full_text (para análise de conteúdo)
df_artifacts_sample = pd.read_parquet(
    PARQUET_FILES['artifacts'],
    # Carregar todas as colunas, mas apenas as primeiras N linhas
    # Nota: Parquet não garante ordem, então isso é uma amostra arbitrária
).head(5000)

print(f'Amostra com texto: {df_artifacts_sample.shape[0]:,} linhas × {df_artifacts_sample.shape[1]} colunas')
print(f'Memória: {df_artifacts_sample.memory_usage(deep=True).sum() / (1024**2):.1f} MB')

## 3. Exploração da Estrutura

### 3.1 Schema e tipos de dados

In [ ]:
print('=== artifacts ===')
display(df_artifacts_sample.dtypes.to_frame('dtype'))
print(f'\nValores nulos:')
null_counts = df_artifacts_sample.isnull().sum()
display(null_counts[null_counts > 0].to_frame('nulos'))

In [ ]:
print('=== repos ===')
display(df_repos.dtypes.to_frame('dtype'))
print(f'\nValores nulos:')
null_counts_repos = df_repos.isnull().sum()
display(null_counts_repos[null_counts_repos > 0].to_frame('nulos'))

In [ ]:
print('=== mining_runs (completa) ===')
display(df_mining)

### 3.2 Primeiros registros

In [ ]:
print('Primeiros registros — artifacts (amostra):')
display(df_artifacts_sample.head(10))

In [ ]:
print('Primeiros registros — repos:')
display(df_repos.head(10))

### 3.3 Estatísticas descritivas

In [ ]:
print('=== Estatísticas Descritivas — artifacts (amostra) ===')
display(df_artifacts_sample.describe(include='all').T)

In [ ]:
print('=== Estatísticas Descritivas — repos ===')
display(df_repos.describe(include='all').T)

## 4. Distribuições Básicas

In [ ]:
# Distribuição de location_class (indica onde o SKILL.md está no repositório)
if 'location_class' in df_artifacts_meta.columns:
    loc_dist = df_artifacts_meta['location_class'].value_counts()
    print('Distribuição de location_class (dataset completo):')
    display(loc_dist)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    loc_dist.plot(kind='barh', ax=ax, color=sns.color_palette('muted'))
    ax.set_title('Distribuição de location_class')
    ax.set_xlabel('Contagem')
    plt.tight_layout()
    plt.show()

In [ ]:
# Distribuição de linguagens dos repositórios
if 'language' in df_repos.columns:
    lang_dist = df_repos['language'].value_counts().head(20)
    print('Top 20 linguagens nos repositórios:')
    display(lang_dist)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    lang_dist.plot(kind='barh', ax=ax, color=sns.color_palette('muted'))
    ax.set_title('Top 20 linguagens dos repositórios com SKILL.md')
    ax.set_xlabel('Contagem')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
# Distribuição de estrelas (log scale)
if 'stargazers_count' in df_repos.columns:
    star_col = 'stargazers_count'
elif 'stars' in df_repos.columns:
    star_col = 'stars'
else:
    star_col = None
    print('Coluna de stars não encontrada. Colunas disponíveis:', list(df_repos.columns))

if star_col:
    fig, ax = plt.subplots(figsize=(10, 5))
    df_repos[star_col].clip(lower=0).apply(lambda x: x + 1).hist(
        bins=50, ax=ax, log=True, color='steelblue', edgecolor='white'
    )
    ax.set_title('Distribuição de Stars dos Repositórios (log scale)')
    ax.set_xlabel('Stars + 1')
    ax.set_ylabel('Contagem (log)')
    ax.set_xscale('log')
    plt.tight_layout()
    plt.show()
    
    print(f'\nEstatísticas de stars:')
    display(df_repos[star_col].describe())

## 5. Conteúdo Textual das Skills

Verificar os campos de texto disponíveis e exemplos de conteúdo.

In [ ]:
# Identificar colunas textuais relevantes
text_cols = [c for c in df_artifacts_sample.columns if any(
    kw in c.lower() for kw in ['text', 'content', 'body', 'front_matter', 'description', 'name']
)]
print(f'Colunas com potencial textual: {text_cols}')

# Mostrar exemplos de conteúdo
for col in text_cols:
    non_null = df_artifacts_sample[col].dropna()
    if len(non_null) > 0:
        avg_len = non_null.astype(str).str.len().mean()
        print(f'\n--- {col} ({len(non_null)} não-nulos, tamanho médio: {avg_len:.0f} chars) ---')
        print(str(non_null.iloc[0])[:500])

In [ ]:
# Identificar a coluna principal de texto
TEXT_COL = None
for candidate in ['full_text', 'content', 'body', 'text']:
    if candidate in df_artifacts_sample.columns:
        TEXT_COL = candidate
        break

if TEXT_COL is None:
    # Fallback: coluna de objeto com maior comprimento médio
    for col in df_artifacts_sample.select_dtypes(include='object').columns:
        avg_len = df_artifacts_sample[col].dropna().astype(str).str.len().mean()
        if avg_len > 200:
            TEXT_COL = col
            break

print(f'Coluna de texto principal: {TEXT_COL}')

if TEXT_COL:
    # Exemplo completo de um SKILL.md
    example = df_artifacts_sample[TEXT_COL].dropna().iloc[0]
    print(f'\n=== Exemplo de skill (primeiros 2000 chars) ===\n')
    print(example[:2000])

## 6. Busca Inicial por Termos de Segurança

Busca por termos relacionados à segurança nos textos das skills para uma primeira noção de prevalência na amostra.

In [ ]:
# Termos de busca relacionados à segurança
SECURITY_KEYWORDS = [
    # Termos gerais de segurança
    'security', 'secure', 'vulnerability', 'exploit',
    'authentication', 'authorization', 'auth',
    'encryption', 'encrypt', 'decrypt',
    # Ataques e vulnerabilidades
    'injection', 'xss', 'csrf', 'sql injection',
    'sanitize', 'sanitization', 'validate', 'validation',
    # Supply chain / dependências
    'dependency', 'supply chain', 'audit',
    # Secrets / credentials
    'secret', 'credential', 'token', 'api key', 'password',
    # Segurança em IA/LLM
    'prompt injection', 'jailbreak', 'guardrail',
    'safety', 'harmful', 'malicious',
    # Práticas e frameworks
    'owasp', 'cve', 'penetration', 'pentest',
    'firewall', 'sandbox', 'permission',
]

print(f'Total de keywords de segurança: {len(SECURITY_KEYWORDS)}')

In [ ]:
if TEXT_COL:
    # Criar coluna de busca (lowercase)
    search_text = df_artifacts_sample[TEXT_COL].fillna('').str.lower()
    
    # Contar ocorrências de cada keyword
    keyword_counts = {}
    for kw in SECURITY_KEYWORDS:
        mask = search_text.str.contains(kw, case=False, na=False)
        keyword_counts[kw] = mask.sum()
    
    keyword_df = pd.DataFrame(
        list(keyword_counts.items()), 
        columns=['keyword', 'count']
    ).sort_values('count', ascending=False)
    
    print(f'Prevalência de keywords na amostra ({len(df_artifacts_sample)} registros):\n')
    display(keyword_df[keyword_df['count'] > 0])
    
    # Visualização
    top_kw = keyword_df[keyword_df['count'] > 0].head(20)
    if len(top_kw) > 0:
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.barplot(data=top_kw, y='keyword', x='count', ax=ax, palette='YlOrRd_r')
        ax.set_title(f'Keywords de Segurança na Amostra (n={len(df_artifacts_sample)})')
        ax.set_xlabel('Ocorrências')
        plt.tight_layout()
        plt.show()
else:
    print('⚠️  Sem coluna de texto para busca.')

In [ ]:
if TEXT_COL:
    # Quantas skills mencionam pelo menos um termo de segurança?
    any_security = search_text.apply(
        lambda t: any(kw in t for kw in SECURITY_KEYWORDS)
    )
    
    n_security = any_security.sum()
    pct_security = any_security.mean()
    
    print(f'Skills com ao menos 1 termo de segurança:')
    print(f'  {n_security} / {len(df_artifacts_sample)} ({pct_security:.1%})')

In [ ]:
# Exibir exemplos de skills que mencionam segurança
if TEXT_COL and n_security > 0:
    security_skills = df_artifacts_sample[any_security].copy()
    
    # Identificar colunas para display
    display_cols = [c for c in [
        'repo', 'repository', 'path', 'name', 'description', 'location_class'
    ] if c in security_skills.columns]
    
    print(f'=== Exemplos de Skills com Termos de Segurança ({len(security_skills)} encontradas) ===\n')
    
    for idx, (_, row) in enumerate(security_skills.head(5).iterrows()):
        print(f'--- Exemplo {idx + 1} ---')
        for col in display_cols:
            val = str(row[col])[:200] if pd.notna(row[col]) else 'N/A'
            print(f'  {col}: {val}')
        # Trecho do texto
        text_preview = str(row[TEXT_COL])[:500] if pd.notna(row[TEXT_COL]) else 'N/A'
        print(f'  {TEXT_COL} (trecho): {text_preview}')
        print()

## 7. Próximos Passos

Após esta exploração inicial, as próximas etapas incluem:

1. **Filtragem em escala:** Aplicar os filtros de segurança ao dataset completo (não apenas à amostra)
2. **Taxonomia de segurança:** Categorizar os tipos de skills de segurança encontradas
3. **Análise de lacunas:** Identificar aspectos de segurança pouco ou não cobertos pelas skills
4. **Qualidade das instruções:** Avaliar se as skills de segurança contêm instruções corretas
5. **Relação com repositórios:** Cruzar com metadados dos repos (linguagem, popularidade, licença)
6. **Análise de supply-chain:** Investigar se skills copiadas introduzem comandos ou acessos maliciosos